In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature=0.1,
)

In [ ]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Any, Type
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.messages import SystemMessage

class StockNewsArgsSchema(BaseModel):
    query: str = Field(description="Company name or stock ticker to search news for")

class StockNewsTool(BaseTool):
    name: str = "StockNewsSearch"
    description: str = "Search for recent financial news about a company or stock."
    args_schema: Type[StockNewsArgsSchema] = StockNewsArgsSchema

    def _run(self, query: str):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(f"{query} stock news")

class StockPriceArgsSchema(BaseModel):
    query: str = Field(description="Stock ticker or company name to search price for")

class StockPriceTool(BaseTool):
    name: str = "StockPriceSearch"
    description: str = "Search for the current stock price of a company."
    args_schema: Type[StockPriceArgsSchema] = StockPriceArgsSchema

    def _run(self, query: str):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(f"{query} stock price today")

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent_type=AgentType.OPENAI_FUNCTIONS,
    tools=[
        StockNewsTool(),
        StockPriceTool(),
    ],
    handle_parsing_errors=True,
    agent_kwargs={
        "system_message": SystemMessage(
            content="""
            You are a conservative personal investment advisor. 
            You always emphasize the risks involved in any investment. 
            You recommend portfolio diversification to minimize risk. 
            You provide advice strictly from a long-term investment perspective, discouraging short-term speculation.
        """
        )
    },
)

agent.run("Should I invest all my savings in Tesla stock?")
